<a href="https://colab.research.google.com/github/DenisBoytsov41/tutors-sentiment-coursework/blob/main/notebooks/03_prepare_labeling_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 03_prepare_labeling_set

Цель ноутбука:
- взять доменно близкие тексты из объединённого корпуса;
- отфильтровать шумные и слишком короткие записи;
- собрать воспроизводимый батч для ручной разметки;
- сохранить:
  - полный master-файл,
  - упрощённый sheet-файл для разметчика,
  - сводки по источникам и ролям.

Источники для ручной разметки:
- `eedi_tutoring`
- `esconv`
- `student_feedback`

Источник, который не идёт в ручную разметку:
- `rusentiment` (у него уже есть готовые метки)

In [46]:
!pip -q install pandas

In [47]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import re
import json
from pathlib import Path
from datetime import datetime

import pandas as pd

In [49]:
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/tutors_sentiment_project")

INTERIM_DIR = DRIVE_PROJECT_ROOT / "02_data_interim"
LABELING_DIR = DRIVE_PROJECT_ROOT / "03_labeling"
REPORTS_DIR = DRIVE_PROJECT_ROOT / "06_reports"

MERGED_DIR = INTERIM_DIR / "merged"
BATCHES_DIR = LABELING_DIR / "batches_for_labeling"
GUIDELINES_DIR = LABELING_DIR / "guidelines"

for d in [BATCHES_DIR, GUIDELINES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DOMAIN_TEXTS_UNIFIED_PATH = MERGED_DIR / "domain_texts_unified.csv"

print("DOMAIN_TEXTS_UNIFIED_PATH =", DOMAIN_TEXTS_UNIFIED_PATH)
print("BATCHES_DIR               =", BATCHES_DIR)

DOMAIN_TEXTS_UNIFIED_PATH = /content/drive/MyDrive/tutors_sentiment_project/02_data_interim/merged/domain_texts_unified.csv
BATCHES_DIR               = /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches_for_labeling


In [50]:
# загрузка корпуса

domain_df = pd.read_csv(DOMAIN_TEXTS_UNIFIED_PATH)

print("Размер корпуса:", domain_df.shape)
print("Колонки:")
print(domain_df.columns.tolist())
domain_df.head()

Размер корпуса: (120140, 12)
Колонки:
['dataset_name', 'source_file', 'conversation_id', 'turn_id', 'speaker_role', 'text', 'gold_label', 'problem_type', 'emotion_type', 'situation', 'split', 'extra_info']


/tmp/ipykernel_628/1068339653.py:3: DtypeWarning: Columns (6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  domain_df = pd.read_csv(DOMAIN_TEXTS_UNIFIED_PATH)


,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,gold_label,problem_type,emotion_type,situation,split,extra_info
0,rusentiment,rusentiment_preselected_posts.csv,NaN,0,NaN,Прорвём информационную блокаду изнутри.,neutral,NaN,NaN,NaN,rusentiment_preselected_posts,NaN
1,rusentiment,rusentiment_preselected_posts.csv,NaN,1,NaN,"Никогда у меня не будет ""одного приложения для...",negative,NaN,NaN,NaN,rusentiment_preselected_posts,NaN
2,rusentiment,rusentiment_preselected_posts.csv,NaN,2,NaN,"Кури-и тебя не укусит злая собака, потому что ...",skip,NaN,NaN,NaN,rusentiment_preselected_posts,NaN
3,rusentiment,rusentiment_preselected_posts.csv,NaN,3,NaN,"Есть 3 типа людей: Умные, которые делают все с...",neutral,NaN,NaN,NaN,rusentiment_preselected_posts,NaN
4,rusentiment,rusentiment_preselected_posts.csv,NaN,4,NaN,мегафон чет накрыло,neutral,NaN,NaN,NaN,rusentiment_preselected_posts,NaN


In [51]:
print("Количество строк по источникам:")
display(domain_df["dataset_name"].value_counts(dropna=False).rename_axis("dataset_name").reset_index(name="count"))

print("Количество строк по источникам и ролям:")
display(domain_df.fillna({"speaker_role": "<none>"}).groupby(["dataset_name", "speaker_role"]).size().reset_index(name="count").sort_values(["dataset_name", "count"], ascending=[True, False]))

Количество строк по источникам:


,dataset_name,count
0,eedi_tutoring,50243
1,esconv,36220
2,rusentiment,31059
3,student_feedback,2618


Количество строк по источникам и ролям:


,dataset_name,speaker_role,count
1,eedi_tutoring,tutor,34081
0,eedi_tutoring,student,16162
2,esconv,seeker,18889
3,esconv,supporter,17331
4,rusentiment,<none>,31059
5,student_feedback,student,2618


In [52]:
# исключаем RuSentiment из ручной разметки

LABELING_SOURCES = ["eedi_tutoring", "esconv", "student_feedback"]

labeling_pool = domain_df[domain_df["dataset_name"].isin(LABELING_SOURCES)].copy()

print("Размер пула для ручной разметки:", labeling_pool.shape)
display(labeling_pool["dataset_name"].value_counts(dropna=False).rename_axis("dataset_name").reset_index(name="count"))

Размер пула для ручной разметки: (89081, 12)


,dataset_name,count
0,eedi_tutoring,50243
1,esconv,36220
2,student_feedback,2618


In [53]:
print("Количество строк по источникам после правки:")
display(labeling_pool["dataset_name"].value_counts(dropna=False).rename_axis("dataset_name").reset_index(name="count"))
print("Количество строк по источникам и ролям после правки:")
display(labeling_pool.fillna({"speaker_role": "<none>"}).groupby(["dataset_name", "speaker_role"]).size().reset_index(name="count").sort_values(["dataset_name", "count"], ascending=[True, False]))

Количество строк по источникам после правки:


,dataset_name,count
0,eedi_tutoring,50243
1,esconv,36220
2,student_feedback,2618


Количество строк по источникам и ролям после правки:


,dataset_name,speaker_role,count
1,eedi_tutoring,tutor,34081
0,eedi_tutoring,student,16162
2,esconv,seeker,18889
3,esconv,supporter,17331
4,student_feedback,student,2618


In [54]:
print("Примеры Eedi:")
display(labeling_pool[labeling_pool["dataset_name"] == "eedi_tutoring"][["speaker_role", "text"]].head(10))

print("Примеры ESConv:")
display(labeling_pool[labeling_pool["dataset_name"] == "esconv"][["speaker_role", "text"]].head(10))

print("Примеры Student Feedback:")
display(labeling_pool[labeling_pool["dataset_name"] == "student_feedback"][["speaker_role", "text"]].head(10))

Примеры Eedi:


,speaker_role,text
31059,tutor,hello how are you?
31060,student,fine
31061,tutor,great!
31062,tutor,Would you like me to look at this question wit...
31063,student,yes
31064,tutor,I'd love to! I'll just read it for a moment
31065,tutor,"first, how do we find 10 less than a number? D..."
31066,student,subtract
31067,tutor,"good, I agree, let's subtract 😊"
31068,tutor,"so, I know it's a large number but the main pa..."


Примеры ESConv:


,speaker_role,text
81302,seeker,Hello
81303,supporter,"Hello, what would you like to talk about?"
81304,seeker,I am having a lot of anxiety about quitting my...
81305,supporter,What makes your job stressful for you?
81306,seeker,I have to deal with many people in hard financ...
81307,supporter,Do you help your clients to make it to a bette...
81308,seeker,"I do, but often they are not going to get back..."
81309,supporter,But you offer them a better future than what t...
81310,seeker,That is true but sometimes I feel like I shoul...
81311,supporter,I can understand that.


Примеры Student Feedback:


,speaker_role,text
117522,student,Throughout the whole semester our focus was on...
117523,student,Duration for the OOSD semester project should ...
117524,student,Both lecturers did a good job on delivering th...
117525,student,Increase the time allocated for design pattern...
117526,student,Useful lecture series
117527,student,Lecture series was good and well organized .
117528,student,Lecture series is very good
117529,student,Lecture series structured very well
117530,student,The module was well structured . But with othe...
117531,student,The lecture series was good but the semester p...


In [55]:
# функции очистки

def normalize_spaces(text):
    if pd.isna(text):
        return None
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text if text else None


def is_noise_text(text):
    if text is None:
        return True

    stripped = text.strip()

    # только спецсимволы / знаки
    if re.fullmatch(r"[\W_]+", stripped):
        return True

    lowered = stripped.lower()

    # типовые слишком короткие служебные ответы
    if lowered in {
        "ok", "okay", "yes", "no", "hi", "hello", "thanks", "thank you",
        "bye", "goodbye"
    }:
        return True

    return False

In [56]:
# расчёт длины и признаков качества текста

labeling_pool["text"] = labeling_pool["text"].map(normalize_spaces)
labeling_pool["char_len"] = labeling_pool["text"].fillna("").str.len()
labeling_pool["word_len"] = labeling_pool["text"].fillna("").str.split().str.len()
labeling_pool["is_noise_text"] = labeling_pool["text"].map(is_noise_text)

print("Описание длины текстов:")
display(labeling_pool[["char_len", "word_len"]].describe())

Описание длины текстов:


,char_len,word_len
count,89081.000000,89081.000000
mean,60.440621,12.334112
std,60.463184,11.732839
min,1.000000,1.000000
25%,22.000000,5.000000
50%,43.000000,9.000000
75%,79.000000,16.000000
max,2530.000000,493.000000


In [57]:
# фильтрация некачественных записей

MIN_CHAR_LEN = 15
MAX_CHAR_LEN = 1200
MIN_WORD_LEN = 3
MAX_WORD_LEN = 220

filtered_pool = labeling_pool[
    labeling_pool["text"].notna()
    & (~labeling_pool["is_noise_text"])
    & (labeling_pool["char_len"] >= MIN_CHAR_LEN)
    & (labeling_pool["char_len"] <= MAX_CHAR_LEN)
    & (labeling_pool["word_len"] >= MIN_WORD_LEN)
    & (labeling_pool["word_len"] <= MAX_WORD_LEN)
].copy()

print("До фильтрации :", labeling_pool.shape)
print("После фильтрации:", filtered_pool.shape)

До фильтрации : (89081, 15)
После фильтрации: (75015, 15)


In [58]:
# что именно убрали

excluded_pool = labeling_pool.loc[~labeling_pool.index.isin(filtered_pool.index)].copy()

print("Исключено строк:", excluded_pool.shape[0])

display(excluded_pool[["dataset_name", "speaker_role", "text", "char_len", "word_len", "is_noise_text"]].head(30))

Исключено строк: 14066


,dataset_name,speaker_role,text,char_len,word_len,is_noise_text
31060,eedi_tutoring,student,fine,4,1,False
31061,eedi_tutoring,tutor,great!,6,1,False
31063,eedi_tutoring,student,yes,3,1,True
31066,eedi_tutoring,student,subtract,8,1,False
31071,eedi_tutoring,student,693,3,1,False
31075,eedi_tutoring,student,18693,5,1,False
31077,eedi_tutoring,student,d,1,1,False
31079,eedi_tutoring,student,yes😀,4,1,False
31081,eedi_tutoring,tutor,Hello again!,12,2,False
31085,eedi_tutoring,student,9,1,1,False


In [59]:
print("После фильтрации по источникам:")
display(filtered_pool["dataset_name"].value_counts(dropna=False).rename_axis("dataset_name").reset_index(name="count"))

print("После фильтрации по источникам и ролям:")
display(filtered_pool.fillna({"speaker_role": "<none>"}).groupby(["dataset_name", "speaker_role"]).size().reset_index(name="count").sort_values(["dataset_name", "count"], ascending=[True, False]))

После фильтрации по источникам:


,dataset_name,count
0,eedi_tutoring,37818
1,esconv,34625
2,student_feedback,2572


После фильтрации по источникам и ролям:


,dataset_name,speaker_role,count
1,eedi_tutoring,tutor,29829
0,eedi_tutoring,student,7989
2,esconv,seeker,17737
3,esconv,supporter,16888
4,student_feedback,student,2572


In [60]:
# квоты на ручную разметку

RANDOM_STATE = 42

QUOTAS = {
    ("eedi_tutoring", "student"): 400,
    ("eedi_tutoring", "tutor"): 400,
    ("esconv", "seeker"): 300,
    ("esconv", "supporter"): 200,
    ("student_feedback", "student"): 200,
}

quota_df = pd.DataFrame([{"dataset_name": k[0], "speaker_role": k[1], "quota": v} for k, v in QUOTAS.items()])

print("Квоты на разметку:")
display(quota_df)
print("Планируемый размер батча:", quota_df["quota"].sum())

Квоты на разметку:


,dataset_name,speaker_role,quota
0,eedi_tutoring,student,400
1,eedi_tutoring,tutor,400
2,esconv,seeker,300
3,esconv,supporter,200
4,student_feedback,student,200


Планируемый размер батча: 1500


In [61]:
# выборка по квотам

sampled_parts = []

for (dataset_name, speaker_role), quota in QUOTAS.items():
    part = filtered_pool[
        (filtered_pool["dataset_name"] == dataset_name)
        & (filtered_pool["speaker_role"].fillna("<none>") == speaker_role)
    ].copy()

    actual_n = min(quota, len(part))
    sampled_part = part.sample(n=actual_n, random_state=RANDOM_STATE)

    sampled_parts.append(sampled_part)

    print(
        f"{dataset_name} / {speaker_role}: "
        f"requested={quota}, available={len(part)}, taken={actual_n}"
    )

labeling_batch = pd.concat(sampled_parts, ignore_index=True)

print("Итоговый размер батча:", labeling_batch.shape)

eedi_tutoring / student: requested=400, available=7989, taken=400
eedi_tutoring / tutor: requested=400, available=29829, taken=400
esconv / seeker: requested=300, available=17737, taken=300
esconv / supporter: requested=200, available=16888, taken=200
student_feedback / student: requested=200, available=2572, taken=200
Итоговый размер батча: (1500, 15)


In [62]:
# признаки приоритета

def mark_priority(row):
    if row["dataset_name"] == "eedi_tutoring":
        return True
    if row["dataset_name"] == "esconv" and row["speaker_role"] == "seeker":
        return True
    return False

labeling_batch["is_domain_priority"] = labeling_batch.apply(mark_priority, axis=1)
labeling_batch["batch_name"] = "labeling_batch_v1"

display(labeling_batch["is_domain_priority"].value_counts().rename_axis("is_domain_priority").reset_index(name="count"))

,is_domain_priority,count
0,True,1100
1,False,400


In [63]:
# подготовка master-файла

labeling_batch = labeling_batch.copy()

labeling_batch["label_item_id"] = [f"LBL_{i:05d}" for i in range(1, len(labeling_batch) + 1)]

labeling_batch["manual_label"] = ""
labeling_batch["labeler_notes"] = ""
labeling_batch["reviewed_flag"] = 0
labeling_batch["adjudicated_label"] = ""

MASTER_COLUMNS = [
    "label_item_id",
    "batch_name",
    "dataset_name",
    "source_file",
    "conversation_id",
    "turn_id",
    "speaker_role",
    "text",
    "problem_type",
    "emotion_type",
    "situation",
    "char_len",
    "word_len",
    "is_domain_priority",
    "manual_label",
    "labeler_notes",
    "reviewed_flag",
    "adjudicated_label",
    "extra_info",
]

labeling_master_df = labeling_batch[MASTER_COLUMNS].copy()

print("Master-файл:")
display(labeling_master_df.head())

Master-файл:


,label_item_id,batch_name,dataset_name,source_file,conversation_id,turn_id,speaker_role,text,problem_type,emotion_type,situation,char_len,word_len,is_domain_priority,manual_label,labeler_notes,reviewed_flag,adjudicated_label,extra_info
0,LBL_00001,labeling_batch_v1,eedi_tutoring,anchored-dialogues/train-00000-of-00001.parquet,6258.0,10,student,Maybe the common denominator would be 126! I t...,NaN,NaN,NaN,51,9,True,,,0,,"{""TalkMovePrediction"": null}"
1,LBL_00002,labeling_batch_v1,eedi_tutoring,anchored-dialogues/train-00000-of-00001.parquet,8171.0,4,student,The question is a bit complicated,NaN,NaN,NaN,33,6,True,,,0,,"{""TalkMovePrediction"": null}"
2,LBL_00003,labeling_batch_v1,eedi_tutoring,anchored-dialogues/train-00000-of-00001.parquet,9408.0,15,student,half the diametre,NaN,NaN,NaN,17,3,True,,,0,,"{""TalkMovePrediction"": null}"
3,LBL_00004,labeling_batch_v1,eedi_tutoring,anchored-dialogues/train-00000-of-00001.parquet,11689.0,4,student,Well do you have a answer,NaN,NaN,NaN,25,6,True,,,0,,"{""TalkMovePrediction"": null}"
4,LBL_00005,labeling_batch_v1,eedi_tutoring,anchored-dialogues/train-00000-of-00001.parquet,1917.0,13,student,so how is the answer not a,NaN,NaN,NaN,26,7,True,,,0,,"{""TalkMovePrediction"": null}"


In [64]:
# подготовка sheet-файла для разметчика

LABELER_COLUMNS = [
    "label_item_id",
    "text",
    "manual_label",
    "labeler_notes",
]

labeling_sheet_df = labeling_master_df[LABELER_COLUMNS].copy()

print("Sheet-файл для разметчика:")
display(labeling_sheet_df.head())

Sheet-файл для разметчика:


,label_item_id,text,manual_label,labeler_notes
0,LBL_00001,Maybe the common denominator would be 126! I t...,,
1,LBL_00002,The question is a bit complicated,,
2,LBL_00003,half the diametre,,
3,LBL_00004,Well do you have a answer,,
4,LBL_00005,so how is the answer not a,,


In [65]:
# сводки по итоговому батчу

labeling_batch_summary = (
    labeling_master_df["dataset_name"]
    .value_counts(dropna=False)
    .rename_axis("dataset_name")
    .reset_index(name="count")
)

labeling_batch_role_summary = (
    labeling_master_df
    .fillna({"speaker_role": "<none>"})
    .groupby(["dataset_name", "speaker_role"])
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "count"], ascending=[True, False])
)

print("Итоговый батч по источникам:")
display(labeling_batch_summary)

print("Итоговый батч по ролям:")
display(labeling_batch_role_summary)

Итоговый батч по источникам:


,dataset_name,count
0,eedi_tutoring,800
1,esconv,500
2,student_feedback,200


Итоговый батч по ролям:


,dataset_name,speaker_role,count
0,eedi_tutoring,student,400
1,eedi_tutoring,tutor,400
2,esconv,seeker,300
3,esconv,supporter,200
4,student_feedback,student,200


In [66]:
# контроль качества батча

print("Размер master:", labeling_master_df.shape)
print("Размер sheet :", labeling_sheet_df.shape)
print("Пустых text  :", labeling_master_df["text"].isna().sum())

duplicate_count = labeling_master_df[["text"]].value_counts().reset_index(name="count")
duplicate_count = duplicate_count[duplicate_count["count"] > 1]

print("Дублей по text в батче:", duplicate_count["count"].sum() if not duplicate_count.empty else 0)

display(labeling_master_df.sample(10, random_state=RANDOM_STATE)[["label_item_id", "dataset_name", "speaker_role", "text", "is_domain_priority"]])

Размер master: (1500, 19)
Размер sheet : (1500, 4)
Пустых text  : 0
Дублей по text в батче: 0


,label_item_id,dataset_name,speaker_role,text,is_domain_priority
1116,LBL_01117,esconv,supporter,I'm sorry to hear that. Why do feel that way?,False
1368,LBL_01369,student_feedback,student,Lecture gave many activities to do in the clas...,False
422,LBL_00423,eedi_tutoring,tutor,"Well, opposite sides have to be parallel.",True
413,LBL_00414,eedi_tutoring,tutor,think about 2/8,True
451,LBL_00452,eedi_tutoring,tutor,"the next number should be bigger, not smaller",True
861,LBL_00862,esconv,seeker,thats nice what the name of the website,True
1063,LBL_01064,esconv,seeker,"No, she is pretty petty about it. It was actua...",True
741,LBL_00742,eedi_tutoring,tutor,Ahh you're doing 4n + 7 not 4n + 3...,True
1272,LBL_01273,esconv,supporter,"yes, Thank you. and i hope you are doing well ...",False
259,LBL_00260,eedi_tutoring,student,I’m confused in this wuestion,True


In [67]:
master_path = BATCHES_DIR / "labeling_batch_v1_master.csv"
sheet_path = BATCHES_DIR / "labeling_batch_v1_sheet.csv"
summary_path = BATCHES_DIR / "labeling_batch_v1_summary.csv"
role_summary_path = BATCHES_DIR / "labeling_batch_v1_role_summary.csv"
excluded_path = BATCHES_DIR / "labeling_batch_v1_excluded_preview.csv"
report_json_path = BATCHES_DIR / "labeling_batch_v1_report.json"

labeling_master_df.to_csv(master_path, index=False, encoding="utf-8-sig")
labeling_sheet_df.to_csv(sheet_path, index=False, encoding="utf-8-sig")
labeling_batch_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
labeling_batch_role_summary.to_csv(role_summary_path, index=False, encoding="utf-8-sig")

excluded_pool[["dataset_name", "speaker_role", "text", "char_len", "word_len", "is_noise_text"]].head(200).to_csv(excluded_path, index=False, encoding="utf-8-sig")

prepare_report = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "domain_pool_size_before_filter": int(labeling_pool.shape[0]),
    "domain_pool_size_after_filter": int(filtered_pool.shape[0]),
    "labeling_batch_size": int(labeling_master_df.shape[0]),
    "sources_used": LABELING_SOURCES,
    "quotas": {f"{k[0]}__{k[1]}": v for k, v in QUOTAS.items()},
    "master_path": str(master_path),
    "sheet_path": str(sheet_path),
}

with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(prepare_report, f, ensure_ascii=False, indent=2)

print("Сохранён master:", master_path)
print("Сохранён sheet :", sheet_path)
print("Сохранены summary-файлы и report.json")

Сохранён master: /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches_for_labeling/labeling_batch_v1_master.csv
Сохранён sheet : /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches_for_labeling/labeling_batch_v1_sheet.csv
Сохранены summary-файлы и report.json


In [68]:
print("=" * 70)
print("Подготовка батча для ручной разметки завершена")
print("=" * 70)
print("Master-файл  :", master_path)
print("Sheet-файл   :", sheet_path)
print("Следующий шаг: ручная разметка labeling_batch_v1_sheet.csv")

Подготовка батча для ручной разметки завершена
Master-файл  : /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches_for_labeling/labeling_batch_v1_master.csv
Sheet-файл   : /content/drive/MyDrive/tutors_sentiment_project/03_labeling/batches_for_labeling/labeling_batch_v1_sheet.csv
Следующий шаг: ручная разметка labeling_batch_v1_sheet.csv
